<a href="https://colab.research.google.com/github/rahmatnug/capstone-cangkringan-ml/blob/main/etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. EXTRACT (Load Data Staging)
# ==========================================
print("Memulai proses Extract...")
# Mengambil data dari hasil generator sebelumnya
df_permintaan = pd.read_csv('mock_permintaan.csv')
df_komoditas = pd.read_csv('mock_komoditas.csv')

# Konversi kolom tanggal menjadi tipe datetime untuk kemudahan sorting temporal
df_permintaan['tanggal_permintaan'] = pd.to_datetime(df_permintaan['tanggal_permintaan'])

# ==========================================
# [SIMULASI DATA KOTOR]
# (Hanya untuk membuktikan ke PM bahwa fungsi ETL bekerja)
# ==========================================
np.random.seed(42)
# 1. Bikin missing values (Nilai Kosong) secara acak sebanyak 5% data
missing_indices = np.random.choice(df_permintaan.index, size=int(len(df_permintaan)*0.05), replace=False)
df_permintaan.loc[missing_indices, 'volume_permintaan'] = np.nan

# 2. Bikin outlier (Anomali) ekstrem
outlier_indices = np.random.choice(df_permintaan.index, size=10, replace=False)
df_permintaan.loc[outlier_indices, 'volume_permintaan'] = 9999.99

# ==========================================
# 2. TRANSFORM (Cleaning, Imputasi, Outlier, Standarisasi)
# ==========================================
print("Memulai proses Transform (Sesuai FR-02 SRS & Use Case)...")

# A. Standarisasi Satuan (Berdasarkan Use Case: Penyetaraan Satuan)
df_merged = pd.merge(df_permintaan, df_komoditas[['id_komoditas', 'satuan']], on='id_komoditas', how='left')
df_merged['satuan'] = df_merged['satuan'].astype(str).str.upper()

# B. Imputasi Missing Values (Berdasarkan SRS FR-02: Tren Temporal)
# Urutkan berdasarkan Poktan, Komoditas, dan Waktu agar interpolasi temporal akurat
df_merged = df_merged.sort_values(by=['id_poktan', 'id_komoditas', 'tanggal_permintaan'])
# Terapkan interpolasi linear. ffill dan bfill untuk jaga-jaga jika NaN ada di baris pertama/terakhir
df_merged['volume_permintaan'] = df_merged.groupby(['id_poktan', 'id_komoditas'])['volume_permintaan'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)
# Bulatkan kembali volume setelah diinterpolasi
df_merged['volume_permintaan'] = df_merged['volume_permintaan'].round(2)

# C. Deteksi dan Eliminasi Anomali/Outlier (Berdasarkan SRS FR-02)
# Menggunakan metode IQR per komoditas
def remove_outliers_iqr(group):
    Q1 = group['volume_permintaan'].quantile(0.25)
    Q3 = group['volume_permintaan'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Hanya simpan data yang masuk batas wajar
    return group[(group['volume_permintaan'] >= lower_bound) & (group['volume_permintaan'] <= upper_bound)]

df_cleaned = df_merged.groupby('id_komoditas', group_keys=False).apply(remove_outliers_iqr)

# D. Update Status Data sesuai rancangan ERD
df_cleaned['status_data'] = 'Cleaned'

# Kembalikan struktur kolom seperti semula (drop kolom satuan bantuan)
df_final = df_cleaned[['id_permintaan', 'id_poktan', 'id_komoditas', 'tanggal_permintaan', 'volume_permintaan', 'status_data']]
# Urutkan kembali berdasarkan ID permintaan agar rapi
df_final = df_final.sort_values(by='id_permintaan').reset_index(drop=True)

# ==========================================
# 3. LOAD (Simpan Data Bersih)
# ==========================================
print("Memulai proses Load...")
df_final.to_csv('cleaned_permintaan.csv', index=False)

print("\n✅ Pipeline ETL Berhasil Dieksekusi!")
print(f"Total baris mentah awal: {len(df_permintaan)}")
print(f"Total baris setelah dibersihkan (outlier dieliminasi): {len(df_final)}")

Memulai proses Extract...
Memulai proses Transform (Sesuai FR-02 SRS & Use Case)...
Memulai proses Load...

✅ Pipeline ETL Berhasil Dieksekusi!
Total baris mentah awal: 1305
Total baris setelah dibersihkan (outlier dieliminasi): 1281


/tmp/ipykernel_1656/2651859911.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = df_merged.groupby('id_komoditas', group_keys=False).apply(remove_outliers_iqr)
